In [66]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

In [67]:
df = pd.read_csv("../datasets/risk_behavior.csv")
df.head()

,Age,Gender,education_level,Prison_type,Prison_security_level,Block_security_level,Incidents_count,Violent_incident_count,escape_incident_count,days_since_last_incidence,total_disciplinary_days,Risk Behaviour
0,58,Male,Primary,Maximum Security,High,High,6,0,1,52,19,Medium
1,61,Male,Secondary,Juvenile,Medium,High,3,0,0,119,4,Low
2,50,Male,Secondary,Remand,Medium,Medium,3,0,0,229,9,Low
3,55,Female,Secondary,Maximum Security,High,High,5,1,0,216,12,Medium
4,39,Male,Secondary,Maximum Security,High,High,8,1,0,194,20,Medium


In [68]:
df.drop_duplicates(inplace=True)

In [69]:
X = df.drop("Risk Behaviour", axis=1)
y = df["Risk Behaviour"]

In [70]:
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

In [71]:
numeric_features = [
    "Age",
    "Incidents_count",
    "Violent_incident_count",
    "escape_incident_count",
    "days_since_last_incidence",
    "total_disciplinary_days"
]

categorical_features = [
    "Gender",
    "education_level",
    "Prison_type",
    "Prison_security_level",
    "Block_security_level"
]

In [72]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [73]:
xgb_model = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    random_state=42,
    eval_metric="mlogloss"
)

In [74]:
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", xgb_model)
])

In [75]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [76]:
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

In [77]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=label_encoder.classes_
    )
)

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.9825

Classification Report
              precision    recall  f1-score   support

        High       0.98      0.98      0.98       879
         Low       0.99      0.99      0.99      1650
      Medium       0.97      0.98      0.98      1471

    accuracy                           0.98      4000
   macro avg       0.98      0.98      0.98      4000
weighted avg       0.98      0.98      0.98      4000


Confusion Matrix
[[ 860    0   19]
 [   0 1630   20]
 [  18   13 1440]]


In [78]:
# Predict risk behavior for a new sample
samples = pd.DataFrame({
    "Age": [24, 40, 55],
    "Gender": ["Male", "Male", "Male"],
    "education_level": ["Primary", "Bachelor's", "Secondary"],
    "Prison_type": ["Minimum Security", "Remand", "Maximum Security"],
    "Prison_security_level": ["Low", "Medium", "High"],
    "Block_security_level": ["Low", "Medium", "High"],
    "Incidents_count": [1, 5, 13],
    "Violent_incident_count": [0, 1, 5],
    "escape_incident_count": [0, 0, 2],
    "days_since_last_incidence": [320, 120, 20],
    "total_disciplinary_days": [2, 12, 40]
})
predicted_risk_behavior = pipeline.predict(samples)
predicted_risk_behavior_label = label_encoder.inverse_transform(predicted_risk_behavior)
print("Predicted Risk Behavior:", predicted_risk_behavior_label)

Predicted Risk Behavior: ['Low' 'Low' 'High']


In [80]:
import joblib
joblib.dump(pipeline, "../../models/risk_behavior_pipeline.pkl")
#kdkdkdkdkdkdkdkd

['../../models/risk_behavior_pipeline.pkl']